# Analyse du marché du streaming vidéo en France (2019–2024)

**Auteur** : Mohammed Zahi — M1 MASERATI, Université Paris-Est Créteil (UPEC)  
**Contact** : z.mohammed91080@gmail.com · [LinkedIn](https://www.linkedin.com/in/mohammed-zahi-04a764343) · [Malt](https://www.malt.fr/profile/mohammedzahi)

---

## Contexte business

Le marché du streaming vidéo en France a explosé depuis 2020. Netflix, Disney+, Amazon Prime, Canal+, OCS, Apple TV+ se livrent une guerre des abonnés intense. Ce projet adopte la posture d'un **Data Analyst au sein d'une Direction Marketing** qui cherche à :

1. **Comprendre la dynamique du marché** — parts de marché, croissance, saisonnalité
2. **Analyser le catalogue** — quel contenu attire les abonnés ?
3. **Modéliser la rétention** — quels facteurs prédisent le churn ?
4. **Produire des KPI actionnables** — recommandations opérationnelles

## Sources de données

| Source | Données | Accès |
|---|---|---|
| **API TMDB** (The Movie Database) | Catalogues, notes, genres, popularité | Gratuite avec clé |
| **Données simulées réalistes** | Abonnés, revenus, churn (basées sur rapports ARCOM/IDATE publics) | Générées en code |

> **Note méthodologique** : Les données d'abonnés sont simulées à partir des rapports publics de l'ARCOM (Autorité de Régulation de la Communication Audiovisuelle) et de l'IDATE DigiWorld. Les ordres de grandeur et tendances reflètent la réalité du marché français.

## Compétences démontrées

- Appel **API REST** avec authentification (TMDB)
- **Simulation de données** réalistes à partir de sources publiques
- **Analyse de parts de marché** et visualisations business
- **Analyse de catalogue** — genres, notes, popularité
- **Modélisation du churn** — régression logistique
- **Dashboard KPI** — tableaux de bord opérationnels
- **Recommandations business** structurées

---

## 1. Installation et configuration

In [ ]:
# Installation (décommenter si nécessaire)
# !pip install pandas numpy matplotlib seaborn requests statsmodels scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import requests
import warnings
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)

# Style global — propre, pro, publication-ready
plt.rcParams.update({
    'figure.figsize': (13, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Palette couleurs plateformes
COLORS = {
    'Netflix':    '#E50914',
    'Canal+':     '#000000',
    'Disney+':    '#113CCF',
    'Amazon':     '#00A8E0',
    'Apple TV+':  '#555555',
    'OCS':        '#FF6B00',
}
BLUE   = '#1a56db'
GREEN  = '#16a34a'
RED    = '#dc2626'
ORANGE = '#d97706'
GRAY   = '#6b7280'

# ─── Clé API TMDB ───
# Obtenir gratuitement sur https://www.themoviedb.org/settings/api
# Remplace la valeur ci-dessous par ta clé
TMDB_API_KEY = 'VOTRE_CLE_API_TMDB_ICI'
TMDB_BASE    = 'https://api.themoviedb.org/3'

def tmdb_get(endpoint, params=None):
    """Appel API TMDB avec gestion d'erreur."""
    p = {'api_key': TMDB_API_KEY, 'language': 'fr-FR'}
    if params:
        p.update(params)
    try:
        r = requests.get(f'{TMDB_BASE}/{endpoint}', params=p, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'API TMDB indisponible ({e}) — données de démonstration utilisées.')
        return None

print('Environnement prêt.')
print('⚠️  Remplace TMDB_API_KEY par ta clé TMDB (gratuite sur themoviedb.org) pour activer les données réelles.')

## 2. Construction de la base de données marché

### 2.1 Données d'abonnés et revenus (simulées à partir des rapports ARCOM 2019–2024)

In [ ]:
# ─── Données marché français — abonnés (millions) ───
# Sources : ARCOM Baromètre audiovisuel 2023, IDATE DigiWorld 2024
# Tendances réelles, valeurs arrondies

annees = list(range(2019, 2025))

abonnes = {
    'Netflix':   [6.7, 7.9, 9.1, 10.2, 10.8, 11.2],
    'Canal+':    [8.1, 8.0, 8.3, 8.5,  8.6,  8.8 ],
    'Disney+':   [0.0, 0.5, 2.1, 3.8,  4.5,  5.0 ],
    'Amazon':    [3.2, 3.8, 4.4, 5.1,  5.6,  6.0 ],
    'Apple TV+': [0.0, 0.2, 0.5, 0.9,  1.3,  1.7 ],
    'OCS':       [2.1, 2.3, 2.2, 2.0,  1.8,  1.5 ],
}

# Prix moyens mensuel (€)
prix_mois = {
    'Netflix':   [10.99, 11.99, 12.99, 13.99, 15.49, 15.49],
    'Canal+':    [19.99, 19.99, 20.99, 20.99, 22.99, 24.99],
    'Disney+':   [0,     6.99,  7.99,  8.99,  8.99,  8.99 ],
    'Amazon':    [5.99,  5.99,  6.99,  6.99,  6.99,  6.99 ],
    'Apple TV+': [0,     4.99,  4.99,  6.99,  9.99,  9.99 ],
    'OCS':       [11.99, 11.99, 11.99, 12.99, 12.99, 12.99],
}

# Construction du DataFrame marché
rows = []
for plateforme in abonnes:
    for i, annee in enumerate(annees):
        abo = abonnes[plateforme][i]
        prix = prix_mois[plateforme][i]
        rows.append({
            'plateforme': plateforme,
            'annee': annee,
            'abonnes_M': abo,
            'prix_mensuel': prix,
            'revenu_annuel_M': round(abo * prix * 12, 1),
        })

df_marche = pd.DataFrame(rows)

# Parts de marché
total_par_annee = df_marche.groupby('annee')['abonnes_M'].sum()
df_marche['part_marche'] = df_marche.apply(
    lambda r: round(r['abonnes_M'] / total_par_annee[r['annee']] * 100, 1), axis=1
)

# Croissance YoY
df_marche = df_marche.sort_values(['plateforme','annee'])
df_marche['croissance_yoy'] = df_marche.groupby('plateforme')['abonnes_M'].pct_change() * 100

print('Base marché construite :')
print(f'  Plateformes : {df_marche["plateforme"].nunique()}')
print(f'  Années      : {df_marche["annee"].min()} – {df_marche["annee"].max()}')
print()
print(df_marche[df_marche['annee'] == 2024][['plateforme','abonnes_M','part_marche','revenu_annuel_M']]\
      .sort_values('part_marche', ascending=False).to_string(index=False))

### 2.2 Données catalogue via API TMDB

In [ ]:
# ─── Catalogue via API TMDB ───
# On récupère les films/séries les plus populaires disponibles en France

def fetch_catalogue(n_pages=5):
    """Récupère les films populaires via TMDB."""
    films = []
    for page in range(1, n_pages + 1):
        data = tmdb_get('movie/popular', {'page': page, 'region': 'FR'})
        if data and 'results' in data:
            films.extend(data['results'])
    return films

def fetch_genres():
    """Récupère le mapping genres TMDB."""
    data = tmdb_get('genre/movie/list')
    if data:
        return {g['id']: g['name'] for g in data.get('genres', [])}
    return {}

print('Récupération du catalogue TMDB...')
films_raw = fetch_catalogue(n_pages=10)
genres_map = fetch_genres()

if films_raw:
    df_catalogue = pd.DataFrame([{
        'titre':        f.get('title', ''),
        'note':         f.get('vote_average', 0),
        'votes':        f.get('vote_count', 0),
        'popularite':   f.get('popularity', 0),
        'date_sortie':  f.get('release_date', ''),
        'genres_ids':   f.get('genre_ids', []),
        'langue':       f.get('original_language', ''),
    } for f in films_raw])

    df_catalogue['date_sortie'] = pd.to_datetime(df_catalogue['date_sortie'], errors='coerce')
    df_catalogue['annee_sortie'] = df_catalogue['date_sortie'].dt.year
    df_catalogue['genre_principal'] = df_catalogue['genres_ids'].apply(
        lambda ids: genres_map.get(ids[0], 'Inconnu') if ids else 'Inconnu'
    )
    df_catalogue = df_catalogue[df_catalogue['votes'] >= 100].copy()
    print(f'  Catalogue récupéré : {len(df_catalogue)} titres')
else:
    # Données de démonstration si API indisponible
    print('  Mode démonstration — génération de données synthétiques réalistes')
    genres_list = ['Action','Comédie','Drame','Thriller','Horreur','Animation','Science-Fiction','Romance','Documentaire']
    langues_list = ['fr','en','es','de','it','ja','ko']
    n = 400
    df_catalogue = pd.DataFrame({
        'titre':          [f'Titre_{i}' for i in range(n)],
        'note':           np.random.normal(6.5, 1.2, n).clip(1, 10).round(1),
        'votes':          np.random.exponential(2000, n).astype(int) + 100,
        'popularite':     np.random.exponential(50, n).round(1),
        'annee_sortie':   np.random.randint(2015, 2025, n),
        'genre_principal':np.random.choice(genres_list, n, p=[.18,.15,.18,.12,.08,.07,.09,.08,.05]),
        'langue':         np.random.choice(langues_list, n, p=[.15,.55,.05,.04,.04,.08,.09]),
    })

print(f'  Genres disponibles : {df_catalogue["genre_principal"].nunique()}')
df_catalogue.head()

### 2.3 Simulation de la base abonnés individuels

In [ ]:
# ─── Base abonnés simulée — 10 000 clients ───
# Chaque ligne = 1 abonné avec son comportement d'usage

N = 10_000
plateformes_list = list(COLORS.keys())
# Pondération selon parts de marché 2024
poids = [0.32, 0.25, 0.14, 0.17, 0.05, 0.07]

df_abonnes = pd.DataFrame({
    'id_client':         range(N),
    'plateforme':        np.random.choice(plateformes_list, N, p=poids),
    'age':               np.random.normal(35, 12, N).clip(18, 75).astype(int),
    'anciennete_mois':   np.random.exponential(18, N).clip(1, 60).astype(int),
    'heures_semaine':    np.random.exponential(7, N).clip(0.5, 40).round(1),
    'nb_profils':        np.random.choice([1,2,3,4,5], N, p=[.30,.35,.20,.10,.05]),
    'nb_genres_vus':     np.random.randint(1, 9, N),
    'a_utilise_mobile':  np.random.choice([0,1], N, p=[.35,.65]),
    'a_utilise_tv':      np.random.choice([0,1], N, p=[.30,.70]),
    'nb_pauses_abonnement': np.random.choice([0,1,2,3], N, p=[.70,.18,.08,.04]),
    'satisfaction_nps':  np.random.normal(7, 2, N).clip(1,10).round(1),
})

# Prix selon la plateforme
prix_ref = {'Netflix':15.49,'Canal+':24.99,'Disney+':8.99,'Amazon':6.99,'Apple TV+':9.99,'OCS':12.99}
df_abonnes['prix_mensuel'] = df_abonnes['plateforme'].map(prix_ref)

# Churn — modèle logistique synthétique
# Facteurs : faible usage, mauvais NPS, ancienneté courte, prix élevé, pauses passées
logit_churn = (
    -0.05 * df_abonnes['heures_semaine']
    - 0.30 * df_abonnes['satisfaction_nps']
    - 0.03 * df_abonnes['anciennete_mois']
    + 0.02 * df_abonnes['prix_mensuel']
    + 0.40 * df_abonnes['nb_pauses_abonnement']
    + 0.10 * (df_abonnes['nb_genres_vus'] < 3).astype(int)
    + np.random.normal(0, 0.5, N)
)
prob_churn = 1 / (1 + np.exp(-logit_churn + 1.5))
df_abonnes['churn'] = (np.random.uniform(0, 1, N) < prob_churn).astype(int)

# Revenu mensuel (0 si churné)
df_abonnes['revenu_mensuel'] = df_abonnes['prix_mensuel'] * (1 - df_abonnes['churn'])

taux_churn_global = df_abonnes['churn'].mean()
print(f'Base abonnés : {N:,} clients simulés')
print(f'Taux de churn global    : {taux_churn_global:.1%}')
print(f'Heures/semaine moyenne  : {df_abonnes["heures_semaine"].mean():.1f}h')
print(f'Ancienneté moyenne      : {df_abonnes["anciennete_mois"].mean():.0f} mois')
df_abonnes.head()

## 3. Dashboard KPI — Vue marché

In [ ]:
# ─── Figure 1 : Dashboard parts de marché ───
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :2])   # Évolution abonnés
ax2 = fig.add_subplot(gs[0, 2])    # Camembert PDM 2024
ax3 = fig.add_subplot(gs[1, :])    # Revenus

# Évolution abonnés
for pf, color in COLORS.items():
    d = df_marche[df_marche['plateforme'] == pf]
    ax1.plot(d['annee'], d['abonnes_M'], color=color, linewidth=2.5,
             marker='o', markersize=5, label=pf)
ax1.set_title('Évolution des abonnements — Marché français (millions)', fontweight='bold')
ax1.set_ylabel('Abonnés (millions)')
ax1.set_xlabel('Année')
ax1.legend(fontsize=8, ncol=2)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}M'))

# Camembert PDM 2024
pdm_2024 = df_marche[df_marche['annee'] == 2024].sort_values('part_marche', ascending=False)
wedge_colors = [COLORS[p] for p in pdm_2024['plateforme']]
wedges, texts, autotexts = ax2.pie(
    pdm_2024['part_marche'], labels=pdm_2024['plateforme'],
    colors=wedge_colors, autopct='%1.1f%%',
    pctdistance=0.75, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in texts:    t.set_fontsize(8)
for at in autotexts:
    at.set_fontsize(7)
    at.set_color('white' if at.get_text() != '' else 'black')
ax2.set_title('Parts de marché 2024', fontweight='bold')

# Revenus annuels
rev_2024 = df_marche[df_marche['annee'] == 2024].sort_values('revenu_annuel_M', ascending=True)
bars = ax3.barh(rev_2024['plateforme'], rev_2024['revenu_annuel_M'],
                color=[COLORS[p] for p in rev_2024['plateforme']], alpha=0.85)
for bar, val in zip(bars, rev_2024['revenu_annuel_M']):
    ax3.text(val + 20, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f} M€', va='center', fontsize=9)
ax3.set_title('Revenus annuels estimés France 2024 (M€)', fontweight='bold')
ax3.set_xlabel('Revenus (millions €)')
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}M€'))

fig.suptitle('Marché du Streaming Vidéo en France — Dashboard 2024',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('fig1_dashboard_marche.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 sauvegardée.')

## 4. Analyse du catalogue

In [ ]:
# ─── Figure 2 : Analyse du catalogue TMDB ───
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Top genres
top_genres = df_catalogue['genre_principal'].value_counts().head(8)
axes[0].barh(top_genres.index[::-1], top_genres.values[::-1],
             color=BLUE, alpha=0.8)
axes[0].set_title('Top 8 genres — Catalogue disponible', fontweight='bold')
axes[0].set_xlabel('Nombre de titres')

# Distribution des notes
axes[1].hist(df_catalogue['note'], bins=20, color=GREEN, alpha=0.8, edgecolor='white')
axes[1].axvline(df_catalogue['note'].mean(), color=RED, linewidth=2,
                linestyle='--', label=f'Moyenne: {df_catalogue["note"].mean():.1f}')
axes[1].set_title('Distribution des notes TMDB', fontweight='bold')
axes[1].set_xlabel('Note moyenne (sur 10)')
axes[1].set_ylabel('Nombre de titres')
axes[1].legend(fontsize=9)

# Note moyenne par genre
note_genre = df_catalogue.groupby('genre_principal')['note'].mean()\
                          .sort_values(ascending=True).tail(8)
colors_genre = [GREEN if n >= 7 else BLUE if n >= 6 else ORANGE for n in note_genre.values]
axes[2].barh(note_genre.index, note_genre.values, color=colors_genre, alpha=0.85)
axes[2].axvline(df_catalogue['note'].mean(), color=GRAY, linewidth=1.5,
                linestyle='--', label='Moyenne globale')
axes[2].set_title('Note moyenne par genre', fontweight='bold')
axes[2].set_xlabel('Note moyenne')
axes[2].set_xlim(0, 10)
for i, (idx, val) in enumerate(note_genre.items()):
    axes[2].text(val + 0.1, i, f'{val:.1f}', va='center', fontsize=8)

plt.suptitle('Analyse du catalogue streaming — Genres et qualité',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2_catalogue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 sauvegardée.')

## 5. Analyse du churn et des KPI clients

In [ ]:
# ─── Figure 3 : KPI churn par plateforme ───
kpi = df_abonnes.groupby('plateforme').agg(
    taux_churn    = ('churn', 'mean'),
    heures_moy    = ('heures_semaine', 'mean'),
    nps_moy       = ('satisfaction_nps', 'mean'),
    anciennete    = ('anciennete_mois', 'mean'),
    nb_clients    = ('id_client', 'count'),
    revenu_total  = ('revenu_mensuel', 'sum'),
).round(2).reset_index()
kpi['taux_churn_pct'] = (kpi['taux_churn'] * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Taux de churn
kpi_sorted = kpi.sort_values('taux_churn_pct', ascending=True)
bar_colors = [COLORS.get(p, GRAY) for p in kpi_sorted['plateforme']]
bars = axes[0].barh(kpi_sorted['plateforme'], kpi_sorted['taux_churn_pct'],
                    color=bar_colors, alpha=0.85)
for bar, val in zip(bars, kpi_sorted['taux_churn_pct']):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
axes[0].set_title('Taux de churn par plateforme (%)', fontweight='bold')
axes[0].set_xlabel('Taux de churn (%)')
axes[0].axvline(kpi['taux_churn_pct'].mean(), color=GRAY, linewidth=1.5,
                linestyle='--', label=f'Moy: {kpi["taux_churn_pct"].mean():.1f}%')
axes[0].legend(fontsize=8)

# Heures de visionnage vs churn
axes[1].scatter(kpi['heures_moy'], kpi['taux_churn_pct'],
                c=[COLORS.get(p, GRAY) for p in kpi['plateforme']],
                s=kpi['nb_clients']/20, alpha=0.8, edgecolors='white', linewidth=1.5)
for _, row in kpi.iterrows():
    axes[1].annotate(row['plateforme'], (row['heures_moy'], row['taux_churn_pct']),
                     fontsize=8, ha='center', va='bottom', xytext=(0, 6),
                     textcoords='offset points')
axes[1].set_title('Usage (h/sem.) vs Taux de churn', fontweight='bold')
axes[1].set_xlabel('Heures de visionnage / semaine')
axes[1].set_ylabel('Taux de churn (%)')
axes[1].text(0.05, 0.95, 'Taille ∝ nb. abonnés',
             transform=axes[1].transAxes, fontsize=8, color=GRAY)

# NPS vs churn
axes[2].scatter(kpi['nps_moy'], kpi['taux_churn_pct'],
                c=[COLORS.get(p, GRAY) for p in kpi['plateforme']],
                s=150, alpha=0.85, edgecolors='white', linewidth=1.5)
for _, row in kpi.iterrows():
    axes[2].annotate(row['plateforme'], (row['nps_moy'], row['taux_churn_pct']),
                     fontsize=8, ha='center', va='bottom', xytext=(0, 6),
                     textcoords='offset points')
z = np.polyfit(kpi['nps_moy'], kpi['taux_churn_pct'], 1)
x_l = np.linspace(kpi['nps_moy'].min(), kpi['nps_moy'].max(), 50)
axes[2].plot(x_l, np.poly1d(z)(x_l), color='black', linewidth=1.5,
             linestyle='--', alpha=0.5)
axes[2].set_title('NPS moyen vs Taux de churn', fontweight='bold')
axes[2].set_xlabel('NPS moyen (satisfaction 1–10)')
axes[2].set_ylabel('Taux de churn (%)')

plt.suptitle('KPI Rétention — Analyse du churn par plateforme',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_kpi_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 sauvegardée.')
print()
print(kpi[['plateforme','taux_churn_pct','heures_moy','nps_moy','anciennete','revenu_total']]\
      .sort_values('taux_churn_pct').to_string(index=False))

## 6. Modélisation du churn — Régression logistique

In [ ]:
# ─── Préparation des features ───
features = [
    'heures_semaine', 'anciennete_mois', 'nb_genres_vus',
    'satisfaction_nps', 'prix_mensuel', 'nb_pauses_abonnement',
    'a_utilise_mobile', 'nb_profils', 'age'
]
target = 'churn'

X = df_abonnes[features].copy()
y = df_abonnes[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ─── Modèle logistique ───
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_sc, y_train)

y_pred      = log_model.predict(X_test_sc)
y_pred_prob = log_model.predict_proba(X_test_sc)[:, 1]
auc         = roc_auc_score(y_test, y_pred_prob)

print(f'=== Modèle de prédiction du churn ===')
print(f'AUC-ROC : {auc:.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Actif','Churné']))

# Importance des variables
coefs = pd.DataFrame({
    'variable': features,
    'coefficient': log_model.coef_[0]
}).sort_values('coefficient')

print('\nCoefficients (standardisés) — impact sur le churn :')
print(coefs.to_string(index=False))

In [ ]:
# ─── Figure 4 : Modèle churn ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coefficients
bar_colors_coef = [RED if c > 0 else GREEN for c in coefs['coefficient']]
axes[0].barh(coefs['variable'], coefs['coefficient'],
             color=bar_colors_coef, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Facteurs de churn — Coefficients logistiques\n(standardisés)',
                  fontweight='bold')
axes[0].set_xlabel('Coefficient (+ = augmente le churn, - = réduit le churn)')
red_patch   = mpatches.Patch(color=RED,   label='Facteur de risque (augmente le churn)')
green_patch = mpatches.Patch(color=GREEN, label='Facteur protecteur (réduit le churn)')
axes[0].legend(handles=[red_patch, green_patch], fontsize=8)

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color=BLUE, linewidth=2.5, label=f'Modèle logistique (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color=GRAY, linewidth=1.5, linestyle='--', label='Aléatoire (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color=BLUE)
axes[1].set_title('Courbe ROC — Prédiction du churn', fontweight='bold')
axes[1].set_xlabel('Taux de Faux Positifs (FPR)')
axes[1].set_ylabel('Taux de Vrais Positifs (TPR)')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.suptitle('Modèle de prédiction du churn — Régression logistique',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_modele_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 sauvegardée.')

## 7. Recommandations business

In [ ]:
# ─── Segmentation des abonnés à risque ───
df_abonnes['proba_churn'] = log_model.predict_proba(scaler.transform(df_abonnes[features]))[:, 1]
df_abonnes['segment_risque'] = pd.cut(
    df_abonnes['proba_churn'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Faible risque', 'Risque modéré', 'Risque élevé']
)

print('=== SEGMENTATION DES ABONNÉS PAR NIVEAU DE RISQUE ===')
seg = df_abonnes.groupby('segment_risque').agg(
    nb_abonnes        = ('id_client', 'count'),
    pct               = ('id_client', lambda x: f"{len(x)/N*100:.1f}%"),
    heures_moy        = ('heures_semaine', 'mean'),
    nps_moy           = ('satisfaction_nps', 'mean'),
    revenu_mensuel_tot= ('revenu_mensuel', 'sum'),
).round(1)
print(seg.to_string())

print()
print('=== RECOMMANDATIONS OPÉRATIONNELLES ===')

coef_top = coefs.sort_values('coefficient')
top_protecteur = coef_top.head(3)['variable'].tolist()
top_risque     = coef_top.tail(3)['variable'].tolist()

print(f"""
1. RÉTENTION — Cibler les {len(df_abonnes[df_abonnes['segment_risque']=='Risque élevé']):,} abonnés
   à risque élevé avec des actions personnalisées.

2. USAGE — Les facteurs qui RÉDUISENT le churn :
   {', '.join(top_protecteur)}
   → Encourager la diversification des genres vus,
     l'usage multi-device et l'augmentation du temps de visionnage.

3. RISQUE — Les facteurs qui AUGMENTENT le churn :
   {', '.join(top_risque)}
   → Surveiller les abonnés avec pauses passées et faible satisfaction.
   → Mettre en place des alertes automatiques pour NPS < 5.

4. VALEUR — Concentrer les efforts sur les segments à fort revenu mensuel.
   AUC du modèle : {auc:.3f} — bonne capacité prédictive pour orienter
   les actions CRM.
""")

In [ ]:
# ─── Figure 5 : Tableau de bord final ───
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Segmentation risque
seg_counts = df_abonnes['segment_risque'].value_counts()
seg_colors = [GREEN, ORANGE, RED]
axes[0].pie(seg_counts, labels=seg_counts.index, colors=seg_colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Segmentation des abonnés\npar niveau de risque churn', fontweight='bold')

# Distribution probabilité de churn
for pf, color in COLORS.items():
    d = df_abonnes[df_abonnes['plateforme'] == pf]['proba_churn']
    axes[1].hist(d, bins=20, alpha=0.5, color=color, label=pf, density=True)
axes[1].set_title('Distribution de la probabilité\nde churn par plateforme', fontweight='bold')
axes[1].set_xlabel('Probabilité de churn')
axes[1].set_ylabel('Densité')
axes[1].legend(fontsize=7, ncol=2)

# Revenu mensuel à risque
rev_risque = df_abonnes.groupby('segment_risque')['revenu_mensuel'].sum().reset_index()
bar_c = [GREEN, ORANGE, RED]
bars = axes[2].bar(rev_risque['segment_risque'], rev_risque['revenu_mensuel'],
                   color=bar_c, alpha=0.85)
for bar, val in zip(bars, rev_risque['revenu_mensuel']):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 500,
                 f'{val:,.0f}€', ha='center', fontsize=9)
axes[2].set_title('Revenu mensuel total\npar segment de risque', fontweight='bold')
axes[2].set_ylabel('Revenu mensuel (€)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}€'))

plt.suptitle('Dashboard Final — Analyse de la valeur et du risque client',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_dashboard_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 sauvegardée.')
print()
print('Figures exportées : fig1–fig5')

---

## À propos

**Mohammed Zahi** — Étudiant M1 MASERATI, UPEC | Freelance Data Analyst  
Python · R · Économétrie · Statistiques · Visualisation · Business Analytics  
z.mohammed91080@gmail.com · [Malt](https://www.malt.fr/profile/mohammedzahi) · [LinkedIn](https://www.linkedin.com/in/mohammed-zahi-04a764343)

**Sources** : ARCOM Baromètre audiovisuel 2023 · IDATE DigiWorld 2024 · TMDB API (themoviedb.org)  
**Note** : Les données d'abonnés sont simulées à partir de rapports publics. Les tendances reflètent la réalité du marché français.